# Preprocessing

In [1]:
DATASET = 'Amazon_Beauty'

REVIEWS_PATH = f'../{DATASET}/reviews_{DATASET[7:]}.json'

INTER_SAVE_PATH = f'../{DATASET}/{DATASET}.inter'
ITEM_SAVE_PATH = f'../{DATASET}/{DATASET}.item'

ITEM_MAPPING_SAVE_PATH = f'../{DATASET}/item_mapping_{DATASET}.json'
USER_MAPPING_SAVE_PATH = f'../{DATASET}/user_mapping_{DATASET}.json'
ITEM_REVERSE_MAPPING_SAVE_PATH = f'../{DATASET}/item_reverse_mapping_{DATASET}.json'
USER_REVERSE_MAPPING_SAVE_PATH = f'../{DATASET}/user_reverse_mapping_{DATASET}.json'

ORGINAL_INTER_PATH = f'../{DATASET}/{DATASET}.inter_orginal'
ORGINAL_ITEM_PATH = f'../{DATASET}/{DATASET}.item_orginal'

DEGENERATE = True
MIN_USER_OCCURRENCE = 5
MIN_ITEM_OCCURRENCE = 5

In [2]:
import pandas as pd
import json
import json5
from tqdm import tqdm

### 1. Generation of .inter file

### 1.1 Reading the 'review' file

In [3]:
data = []
with open(REVIEWS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)

        data.append(
            {
                "user_id:token": record.get("reviewerID"),
                "item_id:token": record.get("asin"),
                "rating:float": record.get("overall"),
                "timestamp:float": record.get("unixReviewTime"),
            }
        )

df = pd.DataFrame(data)
df.head(5)

,user_id:token,item_id:token,rating:float,timestamp:float
0,A39HTATAQ9V7YF,0205616461,5.0,1369699200
1,A3JM6GV9MNOF9X,0558925278,3.0,1355443200
2,A1Z513UWSAAO0F,0558925278,5.0,1404691200
3,A1WMRR494NWEWV,0733001998,4.0,1382572800
4,A3IAAVS479H7M7,0737104473,1.0,1274227200


### 1.2 Filter entries with K-Core algorithm

In [4]:
while True:
    last_shape = df.shape

    df = df.groupby('user_id:token').filter(lambda x: len(x) >= MIN_USER_OCCURRENCE)
    df = df.groupby('item_id:token').filter(lambda x: len(x) >= MIN_USER_OCCURRENCE)

    if df.shape == last_shape:
        break

In [5]:
df.shape

(198502, 4)

### 1.3 Generate mappings

In [6]:
df["user_id:token"], user_index = pd.factorize(df["user_id:token"]) # eg. 42 -> "ASFAFASFASF"
df["item_id:token"], item_index = pd.factorize(df["item_id:token"])

reverse_item_index = {org_id: num for num, org_id in enumerate(item_index)} # eg. "ASFAFASFASF" -> 42
reverse_user_index = {org_id: num for num, org_id in enumerate(user_index)}

### 1.4 Validate the dataset

##### Check for null values

In [7]:
df.isna().sum()

user_id:token      0
item_id:token      0
rating:float       0
timestamp:float    0
dtype: int64

### 1.5 Save the .inter file

In [8]:
df.to_csv(INTER_SAVE_PATH, sep="\t", index=False)

### 1.6 Save the mappings to .json file

In [9]:
with open(ITEM_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(list(item_index), fp=f)

with open(ITEM_REVERSE_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(reverse_item_index, fp=f)

with open(USER_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(list(user_index), fp=f)

with open(USER_REVERSE_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(reverse_user_index, fp=f)

### 2. Generation of .item file

### 2.1 Reading the orginal item file and transforming it

In [10]:
df = pd.read_csv(ORGINAL_ITEM_PATH, delimiter='\t')
df = df[df['item_id:token'].isin(reverse_item_index.keys())]
df['item_id:token'] = df['item_id:token'].map(reverse_item_index)
df.head(5)

,item_id:token,title:token,sales_type:token,sales_rank:float,categories:token_seq,price:float,brand:token
115,0,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,Beauty,10486.0,"'Beauty', 'Makeup', 'Face', 'Concealers & Neut...",5.04,COKA
179,1,Xtreme Brite Brightening Gel 1oz.,Beauty,52254.0,"'Beauty', 'Hair Care', 'Styling Products', 'Cr...",19.99,Xtreme Brite
192,2,Prada Candy By Prada Eau De Parfum Spray 1.7 O...,Beauty,78916.0,"'Beauty', 'Fragrance', ""Women's"", 'Eau de Parfum'",65.86,Prada
555,3,Versace Bright Crystal Eau de Toilette Spray f...,Beauty,764.0,"'Beauty', 'Fragrance', ""Women's"", 'Eau de Toil...",52.33,Versace
587,4,Stella McCartney Stella,Beauty,142503.0,"'Beauty', 'Fragrance', ""Women's"", 'Eau de Parfum'",NaN,NaN


### 2.2 Validate the dataset

##### Check for null values

In [11]:
df.isna().sum()

item_id:token              0
title:token                7
sales_type:token         286
sales_rank:float         286
categories:token_seq       0
price:float              585
brand:token             2098
dtype: int64

### 2.3 Save the .item file

In [12]:
df.to_csv(ITEM_SAVE_PATH, sep='\t', index=False, na_rep='')